In [2]:
from graph_transformer_long_range_niches.tl.wandb import load_and_log
from graph_transformer_long_range_niches.tl.load_config import Config
from graph_transformer_long_range_niches._paths import CFG_FILES, HE22_HUMAN_LUNG_DATA_PATH
from graph_transformer_long_range_niches.model.gnn_transformer import LitGNNTransformer
from graph_transformer_long_range_niches.tl.utils import pad_batch
from graph_transformer_long_range_niches.tl.evaluation import eval_gnntransformer, extract_attention
from graph_transformer_long_range_niches.pl.color_map import CustomColormap
from graph_transformer_long_range_niches.pp.geome_utils import prepare_geome_dataset
from graph_transformer_long_range_niches.config import load_config
from graph_transformer_long_range_niches.pl.attention_matrix import plot_attention_sender_receiver

import warnings
warnings.filterwarnings("ignore")

from torch_geometric.loader import DataLoader

from pathlib import Path
import wandb
import torch
import os

## plotting
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

import scanpy as sc
import squidpy as sq

# Utility functions

In [3]:
def read(data_dir, split):
    """ Read PyG data from artifact folder
    """
    filename = split + ".pt"
    pyg_data = torch.load(os.path.join(data_dir, filename))
    return pyg_data # list with PyG Data objects

In [4]:
# subset adata to validation set
def subset_adata(pyg_datas, adata):
    """
    Subset adata according to the obs_names in a pyg object
    """
    sub_obs_names = []
    for pyg in pyg_datas:
        sub_obs_names.append(pyg.obs_names.numpy().astype(int).astype(str).tolist())
    sub_adata = adata[adata.obs_names.isin(sub_obs_names[0])]
    return sub_adata

# Global parameters

In [5]:
artifact_dir = '/home/icb/francesca.drummer/1-Projects/GT-long-range-niches/docs/notebooks/artifacts'
cfg_path = Path(CFG_FILES, 'pancreas_gnntrans_graph_condition.yaml')

In [6]:
# load config 
cfg = load_config(cfg_path)

## Download data from Wandb

In [7]:
data_name = f"{cfg.dataset.name}_{cfg.dataset.prediction_obs}_{cfg.dataset.library_key}"
model_name = f"{data_name}_{cfg.model.model_type}"
model_path = f'{artifact_dir}/{model_name}_model/{model_name}.ckpt'
#model_path = f'{artifact_dir}/{model_name}_model:v0/pancreas_model.ckpt'
checkpoint = torch.load(model_path)
cfg = checkpoint['hyper_parameters']['cfg']
model = LitGNNTransformer(cfg)
model.load_state_dict(checkpoint['state_dict'])

<All keys matched successfully>

In [8]:
data_dir = f"{artifact_dir}/{model_name}_model" #TODO: adjust to downloaded dataset
pyg_data_val = read(data_dir, 'validation')
pyg_data_val[:3]

[Data(x=[561, 979], edge_index=[2, 340], y=[561, 2], obs_names=[561]),
 Data(x=[754, 979], edge_index=[2, 904], y=[754, 2], obs_names=[754]),
 Data(x=[685, 979], edge_index=[2, 696], y=[685, 2], obs_names=[685])]

In [9]:
adata = read(data_dir, 'adata')
adata

AnnData object with n_obs × n_vars = 108711 × 979
    obs: 'fov', 'Area', 'AspectRatio', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.MembraneStain', 'Max.MembraneStain', 'Mean.PanCK', 'Max.PanCK', 'Mean.GCG', 'Max.GCG', 'Mean.CD3', 'Max.CD3', 'Mean.DAPI', 'Max.DAPI', 'cell_ID', 'condition', 'slide', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_NegPrb', 'log1p_total_counts_NegPrb', 'pct_counts_NegPrb', 'n_genes', 'cell_type_coarse', 'x', 'y', 'sliding_window'
    var: 'NegPrb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells'
    uns: 'log1p', 'spatial', 'one_hot_mappings'
    obsm: 'spatial', 'spatial_fov', 'one_hot'
    layers: 'counts'

# Evaluation on validation data

In [10]:
val_adata = subset_adata(pyg_data_val, adata)
val_adata

View of AnnData object with n_obs × n_vars = 561 × 979
    obs: 'fov', 'Area', 'AspectRatio', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.MembraneStain', 'Max.MembraneStain', 'Mean.PanCK', 'Max.PanCK', 'Mean.GCG', 'Max.GCG', 'Mean.CD3', 'Max.CD3', 'Mean.DAPI', 'Max.DAPI', 'cell_ID', 'condition', 'slide', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_NegPrb', 'log1p_total_counts_NegPrb', 'pct_counts_NegPrb', 'n_genes', 'cell_type_coarse', 'x', 'y', 'sliding_window'
    var: 'NegPrb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells'
    uns: 'log1p', 'spatial', 'one_hot_mappings'
    obsm: 'spatial', 'spatial_fov', 'one_hot'
    layers: 'counts'

In [12]:
eval_loader = DataLoader(pyg_data_val[:1], 1)
for batch in eval_loader:
    #transformer_in, transformer_out, src_padding_mask, index_nodes = eval_gnntransformer(model, batch)
    transformer_in, transformer_out, src_padding_mask, index_nodes = model.evaluation(model, batch)
    print(f'transformer in: {transformer_in.shape}, out: {transformer_out.shape}')
    attn_maps, attn_weights_maps = extract_attention(model.transformer_encoder, transformer_out, src_padding_mask)

transformer in: torch.Size([561, 1, 128]), out: torch.Size([562, 1, 128])
